In [ ]:
import os, subprocess, sys
print("Python:", sys.version)
print("CPUs:", os.cpu_count())
for cmd in (["nvidia-smi"], ["nvcc", "--version"]):
    print("===", " ".join(cmd), "===")
    print(subprocess.run(cmd, text=True, capture_output=True).stdout.strip())

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq && apt-get install -y -qq libfftw3-dev libgsl-dev
rm -rf /content/21cmCU
git clone --depth 1 https://github.com/HainaHuang729/21cmCU.git /content/21cmCU
cd /content/21cmCU
git log -1 --oneline
python3 -m pip install -q cffi setuptools wheel cython pytest \
  click "numpy>=2.0" pyyaml scipy "astropy>=2.0" "h5py>=2.8.0" \
  matplotlib bidict "cosmotile>=0.2.5" attrs tqdm \
  "classy>=3.3.4,<3.4" cyclopts tomlkit "hmf>=3.6.2" deprecation packaging
USE_CUDA=1 CUDA_ARCHS=auto python3 -m pip install --no-build-isolation --no-deps --ignore-requires-python -e .
ls -lh src/py21cmfast/*.so

In [ ]:
import importlib, site, sys
for sp in site.getsitepackages():
    site.addsitedir(sp)
if "/content/21cmCU/src" not in sys.path:
    sys.path.insert(0, "/content/21cmCU/src")
importlib.invalidate_caches()

import py21cmcu
from py21cmfast.c_21cmfast import lib

print("cuda_compiled:", py21cmcu.cuda_compiled())
print("cuda_available:", py21cmcu.cuda_available())
print("cuda_device_count:", py21cmcu.cuda_device_count())
print("get_cuda_devices:", py21cmcu.get_cuda_devices())
assert py21cmcu.cuda_compiled(), "CUDA was not compiled!"
assert py21cmcu.cuda_available(), "No CUDA GPU available!"

required_syms = [
    "p21_cuda_bispectrum_3d",
    "p21_cuda_mfp_bsd",
    "p21_cuda_minkowski_3d",
    "p21_cuda_mgpu_fft_r2c_host",
]
for sym in required_syms:
    assert hasattr(lib, sym), f"Missing symbol: {sym}"
print("All required native CUDA symbols verified:", required_syms)

In [ ]:
%%bash
set -euo pipefail
cd /content/21cmCU
PYTHONPATH=/content/21cmCU/src pytest -v tests/native/test_21cmcu_phase11_batchpool_sbi.py tests/native/test_21cmcu_phase12_topology.py | tee /content/pytest_phase11_12.log

In [ ]:
from pathlib import Path
import json, os, subprocess

bench_code = '''
from __future__ import annotations
import json, os, shutil, subprocess, tempfile, time
from pathlib import Path
import numpy as np
import py21cmcu


def make_ng_field(n: int, box_len: float = 128.0, seed: int = 9501) -> np.ndarray:
    rng = np.random.default_rng(seed)
    shape = (n, n, n)
    white = np.fft.rfftn(rng.normal(0.0, 1.0, size=shape))
    kx = np.fft.fftfreq(n, d=box_len / n) * (2.0 * np.pi)
    ky = np.fft.fftfreq(n, d=box_len / n) * (2.0 * np.pi)
    kz = np.fft.rfftfreq(n, d=box_len / n) * (2.0 * np.pi)
    kmag = np.sqrt(kx[:, None, None] ** 2 + ky[None, :, None] ** 2 + kz[None, None, :] ** 2)
    filt = np.exp(-0.5 * (kmag / 0.45) ** 2)
    filt[0, 0, 0] = 0.0
    g = np.fft.irfftn(white * filt, s=shape, axes=(0, 1, 2))
    g -= np.mean(g)
    g *= 0.25 / np.std(g)
    return (g + 0.25 * (g ** 2 - np.mean(g ** 2))).astype(np.float32)


def make_sphere_xh(n: int, box_len: float = 128.0, r0: float = 28.0) -> np.ndarray:
    c = (np.arange(n) - n / 2.0 + 0.5) * (box_len / n)
    x, y, z = np.meshgrid(c, c, c, indexing="ij")
    return np.where(x * x + y * y + z * z <= r0 * r0, 0.0, 1.0).astype(np.float32)


def make_inputs(seed: int) -> py21cmcu.InputParameters:
    return py21cmcu.InputParameters(
        random_seed=seed, node_redshifts=[8.0]
    ).evolve_input_structs(
        DIM=128,
        HII_DIM=64,
        BOX_LEN=128.0,
        N_THREADS=2,
        SOURCE_MODEL="E-INTEGRAL",
        USE_TS_FLUCT=False,
        RECOMB_MODEL="none",
        USE_EXP_FILTER=False,
        USE_UPPER_STELLAR_TURNOVER=False,
        PHOTON_CONS_TYPE="no-photoncons",
    )


def main() -> None:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
        capture_output=True,
        check=False,
    ).stdout.strip().splitlines()[0]

    # Warm up CUDA kernels
    warm = make_ng_field(64)
    warm_xh = make_sphere_xh(64, r0=16.0)
    _ = py21cmcu.bispectrum_3d(warm, 128.0, config="equilateral", n_bins=8, use_cuda=True)
    _ = py21cmcu.bubble_size_distribution_mfp(warm_xh, 128.0, n_rays=50_000, n_bins=30, seed=42, use_cuda=True)
    _ = py21cmcu.minkowski_functionals_3d(warm, 128.0, n_thresholds=15, standardize=True, use_cuda=True)

    topology_bench = {}
    for n in (128, 256):
        ng = make_ng_field(n)
        xh = make_sphere_xh(n)
        entry = {}

        t0 = time.perf_counter()
        bi_cpu = py21cmcu.bispectrum_3d(ng, 128.0, config="equilateral", n_bins=8, use_cuda=False)
        t_cpu = time.perf_counter() - t0
        t0 = time.perf_counter()
        bi_gpu = py21cmcu.bispectrum_3d(ng, 128.0, config="equilateral", n_bins=8, use_cuda=True)
        t_gpu = time.perf_counter() - t0
        bi_rel = float(np.max(np.abs(bi_gpu["B"] - bi_cpu["B"]) / np.maximum(np.abs(bi_cpu["B"]), 1e-6)))
        entry["bispectrum_3d"] = {
            "cpu_s": round(t_cpu, 4),
            "gpu_s": round(t_gpu, 4),
            "speedup": round(t_cpu / max(t_gpu, 1e-6), 2),
            "max_rel_diff_B": bi_rel,
        }

        t0 = time.perf_counter()
        m_cpu = py21cmcu.bubble_size_distribution_mfp(xh, 128.0, n_rays=200_000, n_bins=30, seed=42, use_cuda=False)
        t_cpu = time.perf_counter() - t0
        t0 = time.perf_counter()
        m_gpu = py21cmcu.bubble_size_distribution_mfp(xh, 128.0, n_rays=200_000, n_bins=30, seed=42, use_cuda=True)
        t_gpu = time.perf_counter() - t0
        mean_r_rel = abs(float(m_gpu["mean_r"]) - float(m_cpu["mean_r"])) / max(abs(float(m_cpu["mean_r"])), 1e-6)
        entry["mfp_bsd_200k"] = {
            "cpu_s": round(t_cpu, 4),
            "gpu_s": round(t_gpu, 4),
            "speedup": round(t_cpu / max(t_gpu, 1e-6), 2),
            "mean_r_rel_diff": mean_r_rel,
        }

        t0 = time.perf_counter()
        mk_cpu = py21cmcu.minkowski_functionals_3d(ng, 128.0, n_thresholds=15, standardize=True, use_cuda=False)
        t_cpu = time.perf_counter() - t0
        t0 = time.perf_counter()
        mk_gpu = py21cmcu.minkowski_functionals_3d(ng, 128.0, n_thresholds=15, standardize=True, use_cuda=True)
        t_gpu = time.perf_counter() - t0
        euler_exact = bool(np.array_equal(mk_gpu["euler"], mk_cpu["euler"]))
        entry["minkowski_3d"] = {
            "cpu_s": round(t_cpu, 4),
            "gpu_s": round(t_gpu, 4),
            "speedup": round(t_cpu / max(t_gpu, 1e-6), 2),
            "euler_exact": euler_exact,
        }
        topology_bench[f"{n}^3"] = entry

    # Warm BatchPool vs Cold run_batch (4 rounds x 2 tasks, 128/64 E-INTEGRAL + SKA1-Low + moments)
    tmp_dir = Path(tempfile.mkdtemp(prefix="21cmcu_colab_"))
    try:
        with py21cmcu.config.use(direc=str(tmp_dir / "cache")):
            rounds = [
                [
                    py21cmcu.BatchTask(
                        task_id=f"r{r}_t{i}",
                        inputs=make_inputs(9600 + r * 10 + i),
                        mode="coeval",
                        out_redshifts=(8.0,),
                        ps_n_bins=6,
                        instrument="SKA1-Low",
                        compute_moments=True,
                    )
                    for i in range(2)
                ]
                for r in range(4)
            ]

            cold_pids = set()
            t0 = time.perf_counter()
            cold_rounds = []
            for tasks in rounds:
                tr0 = time.perf_counter()
                res = py21cmcu.run_batch(
                    tasks,
                    devices=(0,),
                    workers_per_gpu=1,
                    threads_per_worker=2,
                    use_cuda=True,
                    in_process=False,
                )
                assert all(x.ok for x in res), [x.error for x in res]
                cold_pids.update(x.worker_pid for x in res)
                cold_rounds.append(round(time.perf_counter() - tr0, 4))
            cold_total = time.perf_counter() - t0

            warm_pids = set()
            t0 = time.perf_counter()
            warm_rounds = []
            with py21cmcu.BatchPool(
                devices=(0,),
                workers_per_gpu=1,
                threads_per_worker=2,
                use_cuda=True,
                in_process=False,
            ) as pool:
                for tasks in rounds:
                    tr0 = time.perf_counter()
                    res = pool.map(tasks)
                    assert all(x.ok for x in res), [x.error for x in res]
                    warm_pids.update(x.worker_pid for x in res)
                    warm_rounds.append(round(time.perf_counter() - tr0, 4))
            warm_total = time.perf_counter() - t0

            # OnlineSBIDataset zero-disk CUDA tensor verification
            sbi_dir = tmp_dir / "sbi_zero_disk"
            with py21cmcu.config.use(direc=str(sbi_dir)):
                import torch
                sim_field = py21cmcu.SBISimulator(
                    base_inputs=make_inputs(9700),
                    param_names=("F_STAR10_ACG", "F_ESC10_ACG"),
                    mode="coeval",
                    out_redshift=8.0,
                    summary="field",
                    instrument="SKA1-Low",
                    use_cuda=True,
                    lazy_d2h=True,
                    as_torch=True,
                )
                ds = py21cmcu.OnlineSBIDataset(
                    sim_field,
                    {"F_STAR10_ACG": (-1.5, -1.1), "F_ESC10_ACG": (-1.3, -0.8)},
                    n_samples=2,
                    batch_size=2,
                    sampling="sobol",
                    seed=17,
                    as_torch=True,
                )
                th_t, x_t = next(iter(ds))
                torch.cuda.synchronize()
                sbi_info = {
                    "theta_shape": list(th_t.shape),
                    "field_shape": list(x_t.shape),
                    "theta_is_cuda": bool(th_t.is_cuda),
                    "field_is_cuda": bool(x_t.is_cuda),
                    "h5_files_on_disk": len(list(sbi_dir.glob("*.h5"))),
                }

        results = {
            "gpu": gpu_name,
            "cpu_count": os.cpu_count(),
            "topology_gpu_vs_cpu": topology_bench,
            "batch_pool_4x2_128_64": {
                "cold_total_s": round(cold_total, 4),
                "cold_rounds_s": cold_rounds,
                "cold_unique_pids": len(cold_pids),
                "warm_total_s": round(warm_total, 4),
                "warm_rounds_s": warm_rounds,
                "warm_unique_pids": len(warm_pids),
                "speedup": round(cold_total / max(warm_total, 1e-6), 3),
            },
            "online_sbi_zero_disk_cuda": sbi_info,
        }
        Path("/content/21cmcu_colab_gpu_results.json").write_text(
            json.dumps(results, indent=2), encoding="utf-8"
        )
        print("=== BENCHMARK_JSON_START ===")
        print(json.dumps(results, indent=2))
        print("=== BENCHMARK_JSON_END ===")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        py21cmcu.cleanup_cuda()


if __name__ == "__main__":
    main()
'''

Path("/tmp/bench_colab_gpu.py").write_text(bench_code, encoding="utf-8")
subprocess.run(["python3", "/tmp/bench_colab_gpu.py"], env={**os.environ, "PYTHONPATH": "/content/21cmCU/src"}, check=True)

In [ ]:
%%bash
set -euo pipefail
cd /content/21cmCU
PYTHONPATH=/content/21cmCU/src pytest -q tests/native/test_21cmcu_phase*.py | tee /content/pytest_all_native.log